# Clase 107 — Vanishing/exploding gradients

Cuando un gradiente atraviesa muchas capas se **desvanece** (activaciones saturadas → factores < 1 multiplicados muchas veces → ≈ 0) o **explota** (pesos grandes → factores > 1 → ∞). Aquí lo diagnosticamos midiendo la **saturación de la sigmoide**, la **varianza de las activaciones por capa** y la **norma del gradiente por capa**, y mostramos la receta que lo destraba: **He init + ReLU + BatchNorm**.

Requiere: `tensorflow` / `keras` (≥ 3.0), `numpy`, `matplotlib`. Se ejecuta en Colab con GPU.

## 🧠 Intuición previa

**Vanishing / exploding en una frase:** al retropropagar, el gradiente de una capa temprana es un *producto* de muchos factores — las derivadas locales de cada capa que atraviesa. Si esos factores son en promedio **< 1**, el producto se **desvanece** exponencialmente (*vanishing*) y las primeras capas casi no aprenden. Si son **> 1**, el producto **explota** hacia inf/nan (*exploding*). La `sigmoid` satura y aporta factores chiquitos; los pesos gigantes aportan factores grandes. Las clases 108–111 atacan justamente esos factores: inicialización, activación, normalización y clipping.

## 1. La sigmoide satura: su derivada máxima es 0.25

En las colas (`|x| > 5`) la derivada `σ(x)·(1-σ(x))` es ≈ 0. Multiplicar 0.25 diez veces ya aniquila el gradiente.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from tensorflow import keras
from tensorflow.keras import layers
keras.utils.set_random_seed(42)

x = np.linspace(-10, 10, 500)
sig = 1 / (1 + np.exp(-x))
dsig = sig * (1 - sig)                       # derivada de la sigmoide
print("derivada máxima:", round(dsig.max(), 4), "en x =", round(x[dsig.argmax()], 2))
print("(0.25)^10 =", 0.25 ** 10, "→ 10 capas sigmoide casi anulan el gradiente")

## 2. Backprop encadena factores por capa

El gradiente de una capa temprana es un **producto** de derivadas · pesos. Con sigmoide saturada el producto colapsa a 0; con pesos grandes, explota.

In [ ]:
rng = np.random.default_rng(42)
n_capas = 10

# sigmoide saturada: derivadas pequeñas y positivas (< 0.25)
factores_sat = rng.uniform(0.0, 0.25, size=n_capas)
# pesos grandes N(0, 5): factores de magnitud grande
factores_exp = np.abs(rng.normal(0.0, 5.0, size=n_capas))

print("producto |factores| sigmoide (vanishing):", f"{np.prod(factores_sat):.2e}")
print("producto |factores| pesos N(0,5) (exploding):", f"{np.prod(factores_exp):.2e}")

## 3. Varianza de las activaciones por capa

Construimos un MLP profundo que expone la salida de cada capa (modelo multi-salida). Con **sigmoide + Glorot** la varianza se apaga; con **ReLU + He** se mantiene estable.

In [ ]:
def modelo_profundo(activation, initializer, n_capas=20, ancho=100):
    entrada = keras.Input(shape=(784,))
    x = entrada
    salidas = []
    for i in range(n_capas):
        x = layers.Dense(ancho, activation=activation,
                         kernel_initializer=initializer, name=f"dense_{i}")(x)
        salidas.append(x)
    return keras.Model(entrada, salidas)          # una salida por capa

batch = np.random.default_rng(0).normal(size=(256, 784)).astype("float32")
for nombre, act, init in [("sigmoid+Glorot", "sigmoid", "glorot_uniform"),
                          ("relu+He", "relu", "he_normal")]:
    activaciones = modelo_profundo(act, init)(batch)
    stds = [float(a.numpy().std()) for a in activaciones]
    print(f"{nombre:15s} std capa1={stds[0]:.4f} capa10={stds[9]:.4f} capa20={stds[-1]:.4f}")

## 4. Norma del gradiente por capa con `tf.GradientTape`

Diagnóstico directo del vanishing: las capas tempranas de un MLP con sigmoide reciben normas órdenes de magnitud menores que las tardías.

In [ ]:
import tensorflow as tf

modelo = keras.Sequential(
    [keras.Input(shape=(784,))]
    + [layers.Dense(100, activation="sigmoid", name=f"h{i}") for i in range(10)]
    + [layers.Dense(10, activation="softmax", name="out")])

X = tf.constant(np.random.default_rng(1).normal(size=(128, 784)), dtype=tf.float32)
y = tf.constant(np.random.default_rng(1).integers(0, 10, size=128))

with tf.GradientTape() as tape:
    loss = tf.reduce_mean(
        keras.losses.sparse_categorical_crossentropy(y, modelo(X, training=True)))

grads = tape.gradient(loss, modelo.trainable_variables)
for var, g in zip(modelo.trainable_variables, grads):
    if "kernel" in var.name:
        print(f"{var.name:18s} ||grad|| = {tf.norm(g).numpy():.3e}")

## 5. Exploding: init `RandomNormal(stddev=5)`

Pesos iniciales enormes hacen que activaciones y gradientes crezcan sin control. La **norma global** del gradiente se dispara.

In [ ]:
malo = keras.Sequential(
    [keras.Input(shape=(784,))]
    + [layers.Dense(100, activation="relu",
                    kernel_initializer=keras.initializers.RandomNormal(stddev=5.0))
       for _ in range(10)]
    + [layers.Dense(10, activation="softmax")])

with tf.GradientTape() as tape:
    loss = tf.reduce_mean(
        keras.losses.sparse_categorical_crossentropy(y, malo(X, training=True)))
grads = tape.gradient(loss, malo.trainable_variables)
print("norma global del gradiente con init N(0,5):",
      f"{tf.linalg.global_norm(grads).numpy():.3e}")

## 6. La solución: He init + ReLU + BatchNorm

Con esta combinación un MLP de 20 capas entrena estable. `use_bias=False` porque `BatchNormalization` ya aporta el shift `β`.

In [ ]:
def bloque(x, unidades):
    x = layers.Dense(unidades, kernel_initializer="he_normal", use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    return layers.Activation("relu")(x)          # BN antes de la activación

entrada = keras.Input(shape=(784,))
x = entrada
for _ in range(20):
    x = bloque(x, 100)
salida = layers.Dense(10, activation="softmax")(x)
sano = keras.Model(entrada, salida)
sano.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
print("MLP de 20 capas He+ReLU+BN:", sano.count_params(), "parámetros")
# sano.fit(X_tr, y_tr, epochs=5)  # entrenaría estable en Fashion-MNIST

## Ejercicios

1. **Diagnóstico vanishing**: entrená un MLP de 10 capas con `sigmoid` e init por defecto. Compará la norma del gradiente de la primera capa vs la última.
2. **Con ReLU**: repetí el experimento con `relu` + `he_normal`. Mejor, aunque aún heterogéneo.
3. **Exploding**: forzá `RandomNormal(stddev=5)` y observá cómo `loss` llega a `nan` en pocas iteraciones.
4. **Cuatro condiciones**: entrená el MLP profundo bajo (a) sigmoid+Glorot, (b) relu+Glorot, (c) relu+He, (d) relu+He+BN. Graficá `loss` a 20 épocas; la diferencia debe ser dramática.

## Conclusiones

- El gradiente de una capa temprana es un **producto** de factores por capa: <1 muchas veces se desvanece, >1 explota.
- La **sigmoide** tiene derivada máxima 0.25 y satura: no escala en MLPs profundos.
- Diagnosticar con `tf.GradientTape` + `tf.norm` / `tf.linalg.global_norm` por capa es el primer reflejo ante un modelo que no aprende.
- La receta que destrabó el Deep Learning: **He init + ReLU + BatchNorm** (y gradient clipping para RNN/LLMs).
- Los Transformers modernos usan la variante **LayerNorm + residual + GELU + init cuidadoso**.

## ✅ Soluciones de los ejercicios

Diagnóstico de vanishing/exploding gradients y su solución (He + ReLU + BatchNorm). Usamos `tf.GradientTape` para medir `||grad||` por capa. Sin TF se validan por AST.

**Ej. 1 — Diagnóstico vanishing.** MLP de 10 capas `sigmoid`: la primera capa recibe gradientes órdenes de magnitud menores que la última.

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

(Xtr, ytr), _ = keras.datasets.fashion_mnist.load_data()
Xtr = Xtr.reshape(-1, 784).astype("float32") / 255.0
model = keras.Sequential([keras.Input((784,))] +
    [layers.Dense(50, activation="sigmoid") for _ in range(10)] +
    [layers.Dense(10, activation="softmax")])
with tf.GradientTape() as tape:
    loss = keras.losses.SparseCategoricalCrossentropy()(ytr[:256], model(Xtr[:256]))
grads = tape.gradient(loss, model.trainable_variables)
print(f"||grad|| primera capa = {float(tf.norm(grads[0])):.2e}")
print(f"||grad|| ultima capa  = {float(tf.norm(grads[-2])):.2e}")
print("Con sigmoid, la senal se desvanece al retropropagar: la primera capa casi no aprende.")

**Ej. 2 — Mismo experimento con ReLU + He.** Mejora (ReLU no satura en el lado positivo) aunque los gradientes siguen heterogéneos.

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

(Xtr, ytr), _ = keras.datasets.fashion_mnist.load_data()
Xtr = Xtr.reshape(-1, 784).astype("float32") / 255.0
model = keras.Sequential([keras.Input((784,))] +
    [layers.Dense(50, activation="relu", kernel_initializer="he_normal") for _ in range(10)] +
    [layers.Dense(10, activation="softmax")])
with tf.GradientTape() as tape:
    loss = keras.losses.SparseCategoricalCrossentropy()(ytr[:256], model(Xtr[:256]))
grads = tape.gradient(loss, model.trainable_variables)
print("primera:", f"{float(tf.norm(grads[0])):.2e}", " ultima:", f"{float(tf.norm(grads[-2])):.2e}")
print("ReLU + He alivia el vanishing, pero todavia hay disparidad entre capas.")

**Ej. 3 — Exploding.** Con `RandomNormal(stddev=5)` los pesos gigantes hacen que la loss vaya a nan en pocas iteraciones.

In [ ]:
from tensorflow import keras
from tensorflow.keras import layers

(Xtr, ytr), _ = keras.datasets.fashion_mnist.load_data()
Xtr = Xtr.reshape(-1, 784).astype("float32") / 255.0
bad = keras.initializers.RandomNormal(stddev=5)
model = keras.Sequential([keras.Input((784,))] +
    [layers.Dense(50, activation="relu", kernel_initializer=bad) for _ in range(10)] +
    [layers.Dense(10, activation="softmax")])
model.compile(optimizer="sgd", loss="sparse_categorical_crossentropy")
h = model.fit(Xtr, ytr, epochs=1, verbose=0)
print("loss:", h.history["loss"][0], "-> tiende a nan/inf (exploding gradients).")

**Ej. 4 — Norma del gradiente por capa.** Con `tf.GradientTape` medimos `tf.norm(g)` de cada kernel a lo largo del entrenamiento.

In [ ]:
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from tensorflow import keras
from tensorflow.keras import layers

(Xtr, ytr), _ = keras.datasets.fashion_mnist.load_data()
Xtr = Xtr.reshape(-1, 784).astype("float32") / 255.0
model = keras.Sequential([keras.Input((784,))] +
    [layers.Dense(50, activation="relu", kernel_initializer="he_normal") for _ in range(6)] +
    [layers.Dense(10, activation="softmax")])
opt = keras.optimizers.SGD(0.01); loss_fn = keras.losses.SparseCategoricalCrossentropy()
history = []
for _ in range(20):
    with tf.GradientTape() as tape:
        loss = loss_fn(ytr[:128], model(Xtr[:128]))
    grads = tape.gradient(loss, model.trainable_variables)
    history.append([float(tf.norm(g)) for g in grads if g.ndim == 2])   # solo kernels
    opt.apply_gradients(zip(grads, model.trainable_variables))
H = np.array(history)
for layer in range(H.shape[1]):
    plt.plot(H[:, layer], label=f"capa {layer}")
plt.xlabel("step"); plt.ylabel("||grad||"); plt.legend()
plt.title("Norma del gradiente por capa"); plt.show()

**Ej. 5 — Solución sencilla.** He + ReLU + BatchNorm iguala las magnitudes de gradiente entre capas (anticipa las clases 108–111).

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

(Xtr, ytr), _ = keras.datasets.fashion_mnist.load_data()
Xtr = Xtr.reshape(-1, 784).astype("float32") / 255.0

def block(u):
    return [layers.Dense(u, kernel_initializer="he_normal", use_bias=False),
            layers.BatchNormalization(), layers.Activation("relu")]

model = keras.Sequential([keras.Input((784,))] +
    block(50) + block(50) + block(50) + block(50) +
    [layers.Dense(10, activation="softmax")])
with tf.GradientTape() as tape:
    loss = keras.losses.SparseCategoricalCrossentropy()(ytr[:256], model(Xtr[:256], training=True))
grads = tape.gradient(loss, model.trainable_variables)
print("primera:", f"{float(tf.norm(grads[0])):.2e}", " ultima:", f"{float(tf.norm(grads[-2])):.2e}")
print("He + ReLU + BatchNorm: gradientes de magnitud comparable -> el problema desaparece.")